##Environment Setup & Library Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    f1_score,
    average_precision_score,
    confusion_matrix
)
from sklearn.preprocessing import OrdinalEncoder
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

print("Environment setup complete. Libraries imported successfully.")

Environment setup complete. Libraries imported successfully.


##Data Ingestion & Shape Verification

In [3]:
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"
SUB_PATH = "/content/sample_submission.csv"

# Load competition datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUB_PATH)

print(f"Train Dataset Shape: {train_df.shape} (Expected 12,000 rows)")
print(f"Test Dataset Shape:  {test_df.shape} (Expected 8,000 rows)")
print(f"Sample Submission:  {sample_sub.shape} (Expected 8,000 rows)\n")

Train Dataset Shape: (12000, 35) (Expected 12,000 rows)
Test Dataset Shape:  (8000, 34) (Expected 8,000 rows)
Sample Submission:  (8000, 2) (Expected 8,000 rows)



In [4]:
display(train_df.head())

,customer_id,age,tenure_months,number_of_services,data_usage_gb,streaming_hours_monthly,device_age_months,app_sessions_weekly,average_session_minutes,payment_delay_days,...,support_plan,payment_method,promotion_type,streaming_service,priority_support,autopay,paperless_billing,family_plan,international_plan,churn
0,NVT-LGAUGXNSBQFF,50.0,59.0,5.0,5.91,0.0,55.0,14.0,10.4,0.0,...,NovaCare Basic,Credit card (autopay),Loyalty credit,0.0,0.0,1.0,1.0,1.0,0.0,0
1,NVT-KK5E9TRTJYUG,28.0,29.0,4.0,3.03,NaN,3.0,10.0,4.9,0.0,...,NovaCare Premium,Credit card (autopay),No Promotion,0.0,1.0,NaN,1.0,0.0,1.0,0
2,NVT-WC2AWMHHLTTM,59.0,14.0,7.0,86.69,0.0,10.0,10.0,8.1,0.0,...,NovaCare Premium,Bank transfer (autopay),No Promotion,0.0,1.0,1.0,NaN,0.0,1.0,0
3,NVT-WN5PLHPLH7T4,41.0,9.0,1.0,96.32,3.5,2.0,NaN,21.9,0.0,...,No Plan,Bank transfer (autopay),Free month,1.0,0.0,1.0,0.0,0.0,0.0,1
4,NVT-U2ZXSGY3KM9C,55.0,38.0,NaN,3.38,0.0,9.0,11.0,NaN,0.0,...,NaN,Bank transfer (autopay),NaN,0.0,0.0,NaN,1.0,0.0,0.0,0


## Column Schema & Target Distribution Analysis

In [5]:
ID_COL = "customer_id"
TARGET_COL = "churn"

# Define all 33 feature attributes
NUMERICAL_COLS = [
    "age", "tenure_months", "number_of_services", "data_usage_gb",
    "streaming_hours_monthly", "device_age_months", "app_sessions_weekly",
    "average_session_minutes", "payment_delay_days", "late_payment_count",
    "discount_percentage", "monthly_charge", "network_complaints",
    "support_calls_90d", "satisfaction_score", "referral_count", "lifetime_spend"
]

CATEGORICAL_COLS = [
    "customer_segment", "region", "acquisition_channel", "contract_type",
    "plan_tier", "internet_type", "device_type", "support_plan",
    "payment_method", "promotion_type", "streaming_service",
    "priority_support", "autopay", "paperless_billing", "family_plan",
    "international_plan"
]

# Print class distribution
churn_rate = train_df[TARGET_COL].value_counts(normalize=True) * 100
print(f"Target Distribution ('{TARGET_COL}'):")
print(f" - Retained (0): {churn_rate.get(0, 0):.2f}%")
print(f" - Churned  (1): {churn_rate.get(1, 0):.2f}%")

Target Distribution ('churn'):
 - Retained (0): 75.31%
 - Churned  (1): 24.69%


## Minimal Preprocessing & Missing Value Imputation

In [6]:
train_processed = train_df.copy()
test_processed = test_df.copy()

# Impute missing numericals with median
for col in NUMERICAL_COLS:
    if col in train_processed.columns:
        median_val = train_processed[col].median()
        train_processed[col] = train_processed[col].fillna(median_val)
        test_processed[col] = test_processed[col].fillna(median_val)

# Impute missing categoricals with string indicator
for col in CATEGORICAL_COLS:
    if col in train_processed.columns:
        train_processed[col] = train_processed[col].fillna("Missing").astype(str)
        test_processed[col] = test_processed[col].fillna("Missing").astype(str)

print("Imputation completed successfully.")

Imputation completed successfully.


## Categorical Ordinal Encoding

In [7]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train_processed[CATEGORICAL_COLS] = encoder.fit_transform(train_processed[CATEGORICAL_COLS])
test_processed[CATEGORICAL_COLS] = encoder.transform(test_processed[CATEGORICAL_COLS])

# Isolate feature matrices and target array
FEATURE_COLS = NUMERICAL_COLS + CATEGORICAL_COLS
X = train_processed[FEATURE_COLS]
y = train_processed[TARGET_COL]
X_test = test_processed[FEATURE_COLS]

print(f"X Feature Matrix Shape:     {X.shape}")
print(f"X_test Feature Matrix Shape: {X_test.shape}")

X Feature Matrix Shape:     (12000, 33)
X_test Feature Matrix Shape: (8000, 33)


## Validation Strategy Setup

In [8]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Allocate arrays for Out-Of-Fold (OOF) and test set probability predictions
oof_preds = np.zeros(len(train_processed))
test_preds = np.zeros(len(test_processed))

# Default LightGBM hyperparameters targeting ROC-AUC optimization
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': SEED,
    'verbose': -1,
    'n_estimators': 600
}

print(f"Configured {N_SPLITS}-Fold Stratified Cross-Validation.")

Configured 5-Fold Stratified Cross-Validation.


## Model Training & Out-of-Fold Evaluation Loop

In [9]:
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # Instantiate and fit LightGBM model
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # Generate OOF predicted probabilities
    val_probs = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_probs

    # Predict test probabilities (accumulated average across folds)
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS

    # Calculate fold ROC-AUC
    fold_auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(fold_auc)
    print(f"Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")

print(f"\nMean CV ROC-AUC: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}")

Fold 1 ROC-AUC: 0.80532
Fold 2 ROC-AUC: 0.80332
Fold 3 ROC-AUC: 0.81886
Fold 4 ROC-AUC: 0.81415
Fold 5 ROC-AUC: 0.80638

Mean CV ROC-AUC: 0.80960 +/- 0.00591


## Benchmark Metrics & Performance Report

In [10]:
overall_auc = roc_auc_score(y, oof_preds)
overall_logloss = log_loss(y, oof_preds)
overall_prauc = average_precision_score(y, oof_preds)

# Decision metrics at default threshold
binary_oof = (oof_preds >= 0.50).astype(int)
overall_f1 = f1_score(y, binary_oof)

STARTER_BENCHMARK = 0.8100

print("=" * 50)
print("     BASELINE MODEL LOCAL CV EVALUATION     ")
print("=" * 50)
print(f"Overall OOF ROC-AUC Score:  {overall_auc:.5f}")
print(f"Starter Script Benchmark:   ~{STARTER_BENCHMARK:.4f}")
print(f"Performance vs. Benchmark:  {'+' if overall_auc > STARTER_BENCHMARK else ''}{overall_auc - STARTER_BENCHMARK:.5f}")
print("-" * 50)
print(f"PR-AUC (Avg Precision):     {overall_prauc:.5f}")
print(f"Log Loss:                   {overall_logloss:.5f}")
print(f"F1-Score (@ 0.50 cutoff):   {overall_f1:.5f}")
print("=" * 50)
print("\nOOF Confusion Matrix (@ 0.50 cutoff):")
print(confusion_matrix(y, binary_oof))

     BASELINE MODEL LOCAL CV EVALUATION     
Overall OOF ROC-AUC Score:  0.80943
Starter Script Benchmark:   ~0.8100
Performance vs. Benchmark:  -0.00057
--------------------------------------------------
PR-AUC (Avg Precision):     0.58939
Log Loss:                   0.43608
F1-Score (@ 0.50 cutoff):   0.46546

OOF Confusion Matrix (@ 0.50 cutoff):
[[8446  591]
 [1885 1078]]


## Submission Generation & Integrity Checks

In [11]:
submission = pd.DataFrame({
    ID_COL: test_processed[ID_COL],
    TARGET_COL: test_preds
})

# Integrity validations
assert len(submission) == 8000, f"Expected 8,000 rows, got {len(submission)}"
assert submission[TARGET_COL].isnull().sum() == 0, "Submission contains NaN values!"
assert (submission[TARGET_COL] >= 0).all() and (submission[TARGET_COL] <= 1).all(), "Probabilities outside [1]!"

# Output baseline file
OUTPUT_FILE = "baseline_submission.csv"
submission.to_csv(OUTPUT_FILE, index=False)

print(f"Submission saved to '{OUTPUT_FILE}'. Top 5 rows:")
print(submission.head(5))

Submission saved to 'baseline_submission.csv'. Top 5 rows:
        customer_id     churn
0  NVT-E9EXTV4CXKUN  0.177882
1  NVT-QPKCCQHNCJXC  0.072296
2  NVT-5Z8HTRYSMWFH  0.436492
3  NVT-BD5NVDKZ4LZ9  0.353292
4  NVT-WCU3MK4PB7EF  0.063990
